In [ ]:
# This script performs 100 random hyperparameter configurations without random splitting or early stopping
# It was only used for data analysis in Comet, so there is no output as long as the Comet implementation is commented out.

In [ ]:
import random
# from comet_ml import Experiment
import torch
import torch.nn as nn
import pandas as pd
import os
import pickle as pkl
from torch.utils.data import DataLoader
import gc

In [ ]:
# Load data
DATASETS_PATH = os.path.join('..','..','..','data')
TEST_DATASET_PATH = os.path.join(DATASETS_PATH, 'test.pickle')
TRAIN_DATASET_PATH = os.path.join(DATASETS_PATH, 'train.pickle')

with open(TEST_DATASET_PATH, 'rb') as f:
    test_dataset = pkl.load(f)
with open(TRAIN_DATASET_PATH, 'rb') as f:
    train_dataset = pkl.load(f)

In [ ]:
# Dataset class
class SensorDataSet:
    def __init__(self, dataset: pd.DataFrame):
        self.dataset = dataset
    def __getitem__(self, idx):
        data = self.dataset.iloc[idx]
        return torch.tensor(data['sensor_data'], dtype=torch.float32), data['label']
    def __len__(self):
        return len(self.dataset)

In [ ]:
from model import LSTM

In [ ]:
# Evaluate function
def evaluate(model, dataloader, loss_function, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            output = model(x)
            loss = loss_function(output, y)
            total_loss += loss.item() * x.size(0)
            preds = torch.argmax(output, dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return total_loss / total, correct / total

In [ ]:
# Train function
def train(model, dataloader, loss_function, optimizer, epochs, device, experiment=None):
    model.train()
    for epoch in range(epochs):
        total_loss, correct, total = 0.0, 0, 0
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            output = model(x)
            loss = loss_function(output, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * x.size(0)
            preds = torch.argmax(output, dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    # experiment.log_metrics({
    #    "train_loss": total_loss / total,
    #    "train_accuracy": correct / total
    # }, step=epochs)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Run 100 experiments to see the effects of the hyperparameters
for run in range(100):
    # experiment = Experiment(
    #     api_key="",
    #     project_name="brake-failure-detection-runtime-corrected",
    #     auto_output_logging=False,
    #     auto_metric_logging=False,
    #     auto_param_logging=False
    # )
    # experiment.set_name(f"LSTM_run_{run}")

    # Random hyperparameters
    hidden_dim = random.choice([16, 32, 64, 128])
    num_layers = random.choice([1, 2, 3])
    dropout = random.choice([0.0, 0.1, 0.3, 0.5])
    batch_size = random.choice([16, 32, 64])
    learning_rate = random.choice([1e-4, 1e-3, 1e-2])
    weight_decay = random.choice([0.0, 1e-5, 1e-4])
    # hidden_dim, num_layers, dropout, BATCH_SIZE, learning_rate, weight_decay = 64, 1, 0, 32, 0.1, 1e-4 # in case you wanna fix specific ones
    epochs = 10

    # experiment.log_parameters({
    #     "hidden_dim": hidden_dim,
    #     "num_layers": num_layers,
    #     "dropout": dropout,
    #     "batch_size": batch_size,
    #     "learning_rate": learning_rate,
    #     "weight_decay": weight_decay,
    #     "epochs": epochs
    # })

    # DataLoaders
    train_loader = DataLoader(SensorDataSet(train_dataset), batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(SensorDataSet(test_dataset), batch_size=batch_size)

    # Model, loss, optimizer
    model = LSTM(6, hidden_dim, 12, num_layers, dropout).to(device)
    loss_function = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    # Training and testing
    train(model, train_loader, loss_function, optimizer, epochs, device) # If you wanna use comet, add experiment as an argument at the end in this line.
    test_loss, test_acc = evaluate(model, test_loader, loss_function, device)
    # experiment.log_metric("test_loss", test_loss)
    # experiment.log_metric("test_accuracy", test_acc)

    # Save model
    #model_path = f"lstm_model_run_{run}.pth"
    #torch.save(model.state_dict(), model_path)
    #experiment.log_model("lstm_model", model_path)
    #os.remove(model_path)

    # experiment.end()

    # Explicit cleanup
    del model, optimizer, loss_function, train_loader, test_loader
    torch.cuda.empty_cache()
    gc.collect()